# Phishing Model v2 (DistilBERT) - Kaggle Training

**Cara pakai:**
1. Settings > Internet > **ON** (perlu untuk download dataset dari HuggingFace)
2. Settings > Accelerator > **GPU T4 x2** (atau P100)
3. Klik **Save Version > Save and Run All** -> jalan di background
4. Setelah selesai, download `phishing_v2_checkpoint.zip` dari Output tab

> v2 menarik data dari ealvaradob + `insanar/prior-mail-priority` (dua-duanya HuggingFace),
> jadi **tidak perlu upload emails.csv** lagi (Enron off by default).
> Zip output sudah termasuk `threshold.json`, `eval_report.json`, dan `acceptance_report.json`.


In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not available! Go to Settings > Accelerator > GPU T4 x2')


In [ ]:
%%capture
!pip install transformers "datasets>=2.14.0,<3.0.0" accelerate evaluate     wandb sentencepiece pandas regex scikit-learn pyyaml


In [ ]:
import os, sys

!git clone https://github.com/PJK-GM095-PIJAK/prior-mail-model.git
%cd prior-mail-model
!git checkout feat/phishing-distilbert-v2
!git log --oneline -1

sys.path.insert(0, '/kaggle/working/prior-mail-model')


In [ ]:
import shutil
from pathlib import Path

# v2 does NOT need emails.csv (Enron is off; legit class = ealvaradob benign +
# insanar/prior-mail-priority, both from HuggingFace). Copy it only if present.
found = list(Path('/kaggle/input').rglob('emails.csv'))
if found:
    shutil.copy(found[0], Path('emails.csv'))
    print(f'Found emails.csv at {found[0]} (not required for v2, copied anyway)')
else:
    print('No emails.csv found - fine, v2 does not need it.')


In [ ]:
# Wandb offline - JANGAN kasih komentar di baris %env ini
%env WANDB_MODE=offline


In [ ]:
!PYTHONPATH=. python -m src.data.prepare --phishing

from datasets import load_from_disk
ds = load_from_disk('data/processed/phishing')
print('Splits:', {k: ds[k].num_rows for k in ds})


In [ ]:
!PYTHONPATH=. python -m src.training.train_phishing     --config configs/phishing_v2.yaml


In [ ]:
# Selects threshold on val, checks gates on test, writes threshold.json into the
# checkpoint and eval_report.json into eval/results/phishing/.
!PYTHONPATH=. python -m src.eval.eval_phishing --config configs/phishing_v2.yaml


In [ ]:
# Real-world gate: run the model on the curated .eml acceptance set.
!PYTHONPATH=. python -m src.eval.acceptance_phishing --config configs/phishing_v2.yaml


In [ ]:
import json
from pathlib import Path

ckpt = Path('checkpoints/phishing_v2')
print('Checkpoint files:')
for f in sorted(ckpt.glob('*')):
    if f.is_file():
        print(f'  {f.name:<40} {f.stat().st_size/1024/1024:.1f} MB')

for name in ['val_metrics.json', 'threshold.json']:
    p = ckpt / name
    if p.exists():
        print(name + ':', p.read_text())


In [ ]:
import shutil
from pathlib import Path

ckpt = Path('checkpoints/phishing_v2')
res = Path('eval/results/phishing')
for r in ['eval_report.json', 'acceptance_report.json']:
    if (res / r).exists():
        shutil.copy(res / r, ckpt / r)

output_zip = '/kaggle/working/phishing_v2_checkpoint'
shutil.make_archive(output_zip, 'zip', str(ckpt))
print(f'Saved: {output_zip}.zip')
print('Download dari: Kaggle notebook > Output tab > phishing_v2_checkpoint.zip')
